In [ ]:
from __future__ import annotations

import math
from dotenv import load_dotenv
import os
from pydantic import BaseModel
import json
import pandas as pd
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

from pathlib import Path
import time
import csv
import re

import asyncio
import nest_asyncio
import random
from dataclasses import dataclass
from typing import Any

from openai import AsyncOpenAI, APIConnectionError, APIStatusError, RateLimitError
from tqdm.asyncio import tqdm

In [ ]:
import logging
import sys
from io import StringIO
# ─────────────────────────────────────────────
# Настройка директории логов и RUN_ID
# ─────────────────────────────────────────────
from datetime import datetime
from pathlib import Path

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")

# Пробуем подключить Google Drive, если запущено в Colab
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

    # Проверяем, что Drive действительно доступен
    drive_root = Path("/content/drive/MyDrive")
    if not drive_root.exists():
        raise FileNotFoundError(f"Drive примонтирован, но {drive_root} недоступен")

    LOG_DIR = drive_root / "logs"
    USE_DRIVE = True
    print(f"✅ Google Drive подключён.")

except Exception as e:
    # Локальный запуск или Drive недоступен
    LOG_DIR = Path("logs")
    USE_DRIVE = False
    print(f"⚠ Drive недоступен ({e}), логи сохраняются локально в ./logs/")

# ← Создаём папку ЗДЕСЬ, до любых вызовов setup_logger
LOG_DIR.mkdir(parents=True, exist_ok=True)

# Проверка, что папка реально создалась
assert LOG_DIR.exists(), f"Не удалось создать директорию логов: {LOG_DIR}"

print(f"RUN_ID   : {RUN_ID}")
print(f"LOG_DIR  : {LOG_DIR}")
print(f"Папка существует: {LOG_DIR.exists()}")
print(f"USE_DRIVE: {USE_DRIVE}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive подключён.
RUN_ID   : 20260605_211934
LOG_DIR  : /content/drive/MyDrive/logs
Папка существует: True
USE_DRIVE: True


# НАСТРОЙКИ

In [ ]:
load_dotenv(".env")

BASE_URL = os.getenv("BASE_URL")
API_KEY = os.getenv("API_KEY")
MODEL_NAME = "YandexGPT-5-Lite-8B-instruct"
MAX_CONCURRENCY = 256
TEMPERATURE = 0
MAX_TEXT_LEN    = 1500

# ЗАГРУЗКА СЛОВАРЯ

In [ ]:
def load_drug_terms(csv_path="illegal_terms_dictionary_edit.csv"):
    seen: set[tuple[str, str | None]] = set()
    items: list[dict] = []

    with open(csv_path, encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            term = (row.get("normalized_term") or "").strip()
            cat_raw = (row.get("category") or "").strip()
            if not term:
                continue

            term = re.sub(r"\([^)]*\)", "", term)
            if "(" in term or ")" in term or len(term) > 25:
                continue
            term = term.strip(" .,:;").lower()
            if len(term) < 2:
                continue

            category: str | None = cat_raw if cat_raw else None
            key = (term, category)
            if key in seen:
                continue
            seen.add(key)
            items.append({"term": term, "category": category})

    items.sort(key=lambda x: (x["term"], x["category"] or ""))
    return json.dumps(items, ensure_ascii=False)


def load_drug_terms_short(csv_path="illegal_terms_dictionary_edit.csv"):
    seen: set[str] = set()
    items: list[dict] = []

    with open(csv_path, encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            cat_raw = (row.get("category") or "").strip()
            if cat_raw != "drugs":
                continue

            term = (row.get("normalized_term") or "").strip()
            if not term:
                continue

            term = re.sub(r"\([^)]*\)", "", term)
            if "(" in term or ")" in term or len(term) > 25:
                continue
            term = term.strip(" .,:;").lower()
            if len(term) < 2:
                continue

            if term in seen:
                continue
            seen.add(term)
            items.append({"term": term, "category": "drugs"})

    items.sort(key=lambda x: x["term"])
    return json.dumps(items, ensure_ascii=False)


DRUG_TERMS = load_drug_terms("illegal_terms_dictionary_edit.csv")
DRUG_TERMS_SHORT = load_drug_terms_short("illegal_terms_dictionary_edit.csv")

print(f"Загружено терминов в DRUG_TERMS: {len(json.loads(DRUG_TERMS))}")
print(f"Терминов в DRUG_TERMS_SHORT: {len(json.loads(DRUG_TERMS_SHORT))}")

Загружено терминов в DRUG_TERMS: 800
Терминов в DRUG_TERMS_SHORT: 104




# ЗАГРУЗКА ВАЛИДАЦИОННОГО И ТЕСТОВОГО ДАТАСЕТА

In [ ]:

df_val  = pd.read_parquet("val.parquet").reset_index(drop=True)   # ← валидация (подбор гиперпараметров)
df_test = pd.read_parquet("test.parquet").reset_index(drop=True)  # ← финальная оценка

def row_to_input_json(row, idx):
    question = str(row["question"]) if pd.notna(row["question"]) else ""
    answer   = str(row["answer"])   if pd.notna(row["answer"])   else ""
    text = f"Вопрос: {question}\nОтвет: {answer}"
    return {
        "id":   f"{row['session_id']}___{idx}",  # составной ключ
        "text": text[:MAX_TEXT_LEN],
    }

# Входные JSON для классификации берём из val (подбор гиперпараметров)
input_jsons = [row_to_input_json(row, idx) for idx, row in df_val.iterrows()]

print(f"\nВсего записей в val (подбор гиперпараметров): {len(input_jsons)}")
print(f"Уникальных session_id: {len({i['id'] for i in input_jsons})}")
print("\nПример:")
print(json.dumps(input_jsons[0], ensure_ascii=False, indent=2))

assert len({i["id"] for i in input_jsons}) == len(input_jsons), (
    "session_id не уникален в val.parquet! "
    "Нужно использовать составной ключ (session_id + порядковый номер)."
)


Всего записей в val (подбор гиперпараметров): 336
Уникальных session_id: 336

Пример:
{
  "id": "telegram-322189240-onlajn_zakazy-322189240-dJG-4163715281-5452102867.2de8f5e4-9fcc-c2b2-2acc-63fd3ca1803b___0",
  "text": "Вопрос: /newNode_7;Альфа-ПВП VHQ+ТОП 0.5гр.280грн\nОтвет: Избран продукт: Альфа-ПВП VHQ+ ТОП Прозрачные крисы 0.5 гр.\nКоротко о товаре: Всеми любимая и знакомая Альфа пвп. Очень мощные, прозрачные криссталы, высокого качества!!!\nЦена: 280 грн.\nВыберите подходящий район:"
}


# Загрузка тренировочного датасета и выбор примеров для промпта

In [ ]:
df_train = pd.read_parquet("train.parquet").reset_index(drop=True)

# Задаём фиксированный seed для воспроизводимости
SEED_VALUE = 42
random.seed(SEED_VALUE)

# t — количество случайных примеров, которые нужно выбрать. Массив для тестирования: [2, 4, 8, 16, 32]) примеров
T_GRID = [2, 4, 8, 16, 32]


# ─────────────────────────────────────────────
# Подготовка данных
# ─────────────────────────────────────────────
def row_to_train_json(row, idx):
    question = str(row["question"]) if pd.notna(row["question"]) else ""
    answer   = str(row["answer"])   if pd.notna(row["answer"])   else ""
    text = f"Вопрос: {question}\nОтвет: {answer}"
    return {
        "id":    f"{row['session_id']}___{idx}",
        "text":  text[:MAX_TEXT_LEN],
        "label": int(row["from_illegal_account"]),  # 0 = legal, 1 = illegal
    }

examples_jsons = [row_to_train_json(row, idx) for idx, row in df_train.iterrows()]

examples_id    = [item["id"]    for item in examples_jsons]
examples_text  = [item["text"]  for item in examples_jsons]
examples_label = [item["label"] for item in examples_jsons]

print(f"\nВсего записей в train : {len(examples_id)}")
print(f"Уникальных session_id : {len(set(examples_id))}")
print(f"Legal   (0)           : {examples_label.count(0)}")
print(f"Illegal (1)           : {examples_label.count(1)}")

# Проверка уникальности ID
assert len(set(examples_id)) == len(examples_id), (
    "session_id не уникален в train.parquet! "
    "Нужно использовать составной ключ (session_id + порядковый номер)."
)

# Индексы по классам
legal_indices   = [i for i, lbl in enumerate(examples_label) if lbl == 0]
illegal_indices = [i for i, lbl in enumerate(examples_label) if lbl == 1]

# ─────────────────────────────────────────────
# Вспомогательная функция вывода
# ─────────────────────────────────────────────
def print_samples(indices: list, title: str):
    print(f"\n{'='*60}")
    print(f"  {title}")
    print(f"{'='*60}")
    for idx in sorted(indices):
        label_str = "illegal" if examples_label[idx] == 1 else "legal"
        print(f"\n[{label_str}]")
        #ID: {examples_id[idx]}")
        print(examples_text[idx])
        print("-" * 40)
# ─────────────────────────────────────────────
# Вариант 1 — t рандомных примеров
# ─────────────────────────────────────────────
def sample_random(t: int) -> list:
    return random.sample(range(len(examples_id)), t)

# ─────────────────────────────────────────────
# Вариант 2 — ровно t примеров, соотношение 1:1
#
# Идея: берём t//2 illegal + t//2 legal,
# остаток (t % 2) добираем из случайного класса.
# Итого всегда ровно t.
# ─────────────────────────────────────────────
def sample_balanced(t: int) -> list:
    n_illegal = t // 2
    n_legal   = t // 2
    remainder = t % 2  # 0 или 1

    sampled_legal   = random.sample(legal_indices,   n_legal)
    sampled_illegal = random.sample(illegal_indices, n_illegal)

    # Остаток добираем из случайного класса
    if remainder:
        extra_pool = (
            [i for i in legal_indices   if i not in sampled_legal] +
            [i for i in illegal_indices if i not in sampled_illegal]
        )
        sampled_illegal += random.sample(extra_pool, remainder)

    result = sampled_legal + sampled_illegal
    assert len(result) == t, f"balanced: ожидалось {t}, получено {len(result)}"
    return result

# Вариант 3 — ровно t примеров, соотношение 2:1 (legal:illegal)
#
# Идея: вычисляем n_illegal = round(t / 3),
# n_legal = t - n_illegal.
# Это даёт соотношение ~2:1 без потери примеров.
# ─────────────────────────────────────────────
def sample_2to1(t: int) -> list:
    # Целевое соотношение legal:illegal = 2:1
    # n_illegal ≈ t/3, но не менее 1 (чтобы класс не выпал)
    n_illegal = max(1, round(t / 3))
    n_legal   = t - n_illegal  # всегда t - n_illegal, итого ровно t

    # Защита: не запрашиваем больше, чем есть в пуле
    n_illegal = min(n_illegal, len(illegal_indices))
    n_legal   = min(n_legal,   len(legal_indices))
    actual_t  = n_legal + n_illegal  # может быть < t только если пул мал

    sampled_legal   = random.sample(legal_indices,   n_legal)
    sampled_illegal = random.sample(illegal_indices, n_illegal)

    result = sampled_legal + sampled_illegal
    assert len(result) == actual_t
    return result




Всего записей в train : 748
Уникальных session_id : 748
Legal   (0)           : 182
Illegal (1)           : 566




# ПРОМПТЫ

In [ ]:
SYSTEM_PROMPT = "Ты - помощник по классификации текста для задачи модерации на предмет упоминания наркотиков."
prompt_d = """РОЛЬ:
Ты — высокоточная система бинарной классификации (NLP-модель), предназначенная для модерации сообщений Telegram. Твоя цель — максимально точно (с приоритетом на высокий F1-score) определять наличие упоминаний наркотических и психоактивных веществ.

ЗАДАЧА:
Определи, содержит ли текст сообщения упоминания наркотиков или связанной с ними деятельности.

ФОРМАТ ВХОДА:
JSON с полями:
- id: идентификатор сообщения
- text: текст сообщения (единственный источник анализа)

ФОРМАТ ВЫХОДА:
Строго JSON: {{"has_drug_mention": true | false}}

ОПРЕДЕЛЕНИЕ КЛАССА true:
Ставь true, если выполнено ХОТЯ БЫ ОДНО из условий:

1. Прямое упоминание наркотиков:
   - каннабис, марихуана, гашиш, кокаин, героин, амфетамин, метамфетамин, экстази, LSD и т.д.
   - любые термины из словаря ниже (полное или частичное совпадение по корню)

2. Прямое упоминание сущностей, связанных с наркотиками и наркоторговлей:
   - обменник, клад, кладмен, фасовка, закладка и т.д.

2. Сленг, жаргон, эвфемизмы:
   - шишки, травка, соль (в наркотическом контексте), меф, спиды, колёса, бошки и т.д.
   - английский сленг: weed, coke, meth, molly, acid и т.д.

3. Намеренно искажённые слова, ососбенно в названиях каналов и ботов через @:
   - замены символов: м@рuху@на, к0к@ин, мефедр0н
   - добавление лишних символов: DeaIler
   - пробелы/разделители: "м е ф", "к о к с"
   - транслит: marikhuana, geroin, mefedron

4. Контекст действий:
   - покупка, продажа, обмен, доставка, закладки
   - употребление, хранение, производство
   - поиск: "где взять", "купить", "есть ли", "ищу"

5. Подозрительные аббревиатуры и одиночные буквы латиницей в качестве вопроса:
   - Bbgg, Sh, I, CV GK j

6. Упоминания криптокошельков и криптовалют

7. Фразы с двойным дном и иносказания:
   -  Главное не забывать: счастье — это когда ты нашёл, а тебя нет!

8. Косвенные сигналы:
   - эмодзи: 💊 🌿 🍁 ❄️ 🔥 🚬 💉
   - сочетание нейтральных слов с подозрительным контекстом

9. Частично неоднозначные случаи:
   - если есть разумное подозрение на наркотический контекст → true

СЛОВАРЬ ТЕРМИНОВ НАРКОТИЧЕСКОЙ ТЕМАТИКИ (список объектов с term и category — используй как опорный список):
{DRUG_TERMS}

ОПРЕДЕЛЕНИЕ КЛАССА false:
Ставь false, если:

1. Упоминания отсутствуют полностью
2. Слова-омонимы используются в бытовом значении:
   - "соль", "сахар", "таблетки" без контекста наркотиков
3. Лекарства:
   - если это медицинский контекст без признаков злоупотребления
4. Явная ирония или метафоры:
   - "я подсел на кофе как на наркотик"
5. Общие разговоры без связи с наркотиками

ПРИОРИТЕТ ПРИНЯТИЯ РЕШЕНИЯ (ВАЖНО ДЛЯ F1):
1. Контекст > отдельные слова
2. Словарь (DRUG_TERMS) > эвристики
3. Лучше false, чем ложный true ТОЛЬКО если нет достаточных сигналов
4. Но при слабом, но логичном подозрении → true

АЛГОРИТМ (внутренний):
1. Нормализуй текст:
   - приведи к нижнему регистру
   - убери спецсимволы
   - учти замены символов и транслит
2. Проверь совпадения со словарём DRUG_TERMS (term и его category помогут понять контекст: drugs/drug_trade/drug_effect/instruments/general/crypto_payment)
3. Проверь сленг и искажения
4. Проанализируй контекст (действия + объекты)
5. Учти эмодзи
6. Прими бинарное решение

ОГРАНИЧЕНИЯ:
- Используй ТОЛЬКО поле text
- НЕ добавляй объяснений
- НЕ добавляй новых полей
- НЕ изменяй структуру JSON
- Ответ ДОЛЖЕН быть валидным JSON

ПРИМЕР:

ВХОД:
{{
  "id": "abc__0",
  "text": "где купить меф?"
}}

ВЫХОД:
{{"has_drug_mention": true}}

ПРИМЕРЫ ИЗ ОБУЧАЮЩЕЙ ВЫБОРКИ:
{FEW_SHOT_EXAMPLES}

ВХОДНОЙ JSON ДЛЯ КЛАССИФИКАЦИИ:
{INPUT_JSON}
"""

In [ ]:
# Запустите один раз после определения prompt_d
KNOWN_PLACEHOLDERS = {"FEW_SHOT_EXAMPLES", "DRUG_TERMS", "INPUT_JSON"}

def escape_unknown_placeholders(text: str, known: set) -> str:
    """Экранирует {VAR} если VAR не в known."""
    def replacer(m):
        key = m.group(1)
        return f"{{{{{key}}}}}" if key not in known else m.group(0)
    return re.sub(r'\{(\w+)\}', replacer, text)

prompt_d = escape_unknown_placeholders(prompt_d, KNOWN_PLACEHOLDERS)

# Проверка
remaining = re.findall(r'\{(\w+)\}', prompt_d)
print("Оставшиеся плейсхолдеры:", remaining)
# Должно быть: ['FEW_SHOT_EXAMPLES', 'DRUG_TERMS', 'INPUT_JSON']

Оставшиеся плейсхолдеры: ['DRUG_TERMS', 'FEW_SHOT_EXAMPLES', 'INPUT_JSON']


In [ ]:
# ─────────────────────────────────────────────
# Построитель few-shot блока из списка индексов
# ─────────────────────────────────────────────
def build_few_shot_block(indices: list) -> str:
    """
    Формирует строку с примерами для вставки в промпт.
    Метка берётся из реального examples_label[idx].
    """
    blocks = []
    for idx in sorted(indices):
        label = bool(examples_label[idx] == 1)  # ← реальная метка
        label_str = "ILLEGAL" if label else "LEGAL"

        example_input = json.dumps(
            {"text": examples_text[idx]},
            ensure_ascii=False,
            indent=2,
        )
        example_output = json.dumps(
            {"has_drug_mention": label},          # ← без id, реальная метка
            ensure_ascii=False,
            indent=2,
        )
        blocks.append(
            f"[{label_str}]\n"
            f"ПРИМЕР ВХОДА:\n{example_input}\n\n"
            f"ПРИМЕР ВЫХОДА:\n{example_output}"
        )
    return "\n\n" + ("\n\n" + "─" * 40 + "\n\n").join(blocks) + "\n"


# ─────────────────────────────────────────────
# Построитель сообщений
# sample_fn — функция выборки, вызывается заново для каждого item
# ─────────────────────────────────────────────
def build_messages_d(item, sample_fn=None, use_dict: bool = True, t: int = 10):
    if sample_fn is not None:
        indices = sample_fn(t)
        few_shot_block = build_few_shot_block(indices)
    else:
        few_shot_block = "(примеры не используются)"

    # DRUG_TERMS — полный словарь, используется в теле промпта как {DRUG_TERMS}
    # DRUG_TERMS_SHORT — короткий словарь (только drugs), в промпте НЕ используется
    # → передаём DRUG_TERMS, убираем DRUG_TERMS_SHORT
    drug_terms_value = DRUG_TERMS if use_dict else "(словарь не используется)"

    user_content = prompt_d.format(
        FEW_SHOT_EXAMPLES=few_shot_block,
        DRUG_TERMS=drug_terms_value,          # ✅ совпадает с {DRUG_TERMS} в промпте
        INPUT_JSON=json.dumps(item, ensure_ascii=False, indent=2),
    )
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_content},
    ]

# ─────────────────────────────────────────────
# Генерация PROMPT_VARIANTS
# Структура имени: {prompt}_{strategy}_{dict}_{t}exmpls
# ─────────────────────────────────────────────

# Базовые конфигурации: (strategy_name, sample_fn, use_dict)
BASE_CONFIGS = [
    ("random",   sample_random,   False),
    ("balanced", sample_balanced, False),
    ("2to1",     sample_2to1,     False),
    ("random",   sample_random,   True),
    ("balanced", sample_balanced, True),
    ("2to1",     sample_2to1,     True),
]

def make_variant_name(strategy: str, use_dict: bool, t: int) -> str:
    dict_suffix = "with_dict" if use_dict else "wo_dict"
    return f"prompt_d_{strategy}_few_shot_{dict_suffix}_{t}exmpls"

# Собираем словарь вариантов
# Используем замыкание через default-аргументы, чтобы избежать
# проблемы позднего связывания (late binding) в цикле
PROMPT_VARIANTS: dict[str, callable] = {}

for _strategy, _sample_fn, _use_dict in BASE_CONFIGS:
    for _t in T_GRID:
        _name = make_variant_name(_strategy, _use_dict, _t)
        PROMPT_VARIANTS[_name] = (
            lambda x, fn=_sample_fn, ud=_use_dict, t_val=_t:
                build_messages_d(x, sample_fn=fn, use_dict=ud, t=t_val)
        )

# Проверка
print(f"Всего вариантов промптов: {len(PROMPT_VARIANTS)}")
for name in PROMPT_VARIANTS:
    print(f"  {name}")

Всего вариантов промптов: 30
  prompt_d_random_few_shot_wo_dict_2exmpls
  prompt_d_random_few_shot_wo_dict_4exmpls
  prompt_d_random_few_shot_wo_dict_8exmpls
  prompt_d_random_few_shot_wo_dict_16exmpls
  prompt_d_random_few_shot_wo_dict_32exmpls
  prompt_d_balanced_few_shot_wo_dict_2exmpls
  prompt_d_balanced_few_shot_wo_dict_4exmpls
  prompt_d_balanced_few_shot_wo_dict_8exmpls
  prompt_d_balanced_few_shot_wo_dict_16exmpls
  prompt_d_balanced_few_shot_wo_dict_32exmpls
  prompt_d_2to1_few_shot_wo_dict_2exmpls
  prompt_d_2to1_few_shot_wo_dict_4exmpls
  prompt_d_2to1_few_shot_wo_dict_8exmpls
  prompt_d_2to1_few_shot_wo_dict_16exmpls
  prompt_d_2to1_few_shot_wo_dict_32exmpls
  prompt_d_random_few_shot_with_dict_2exmpls
  prompt_d_random_few_shot_with_dict_4exmpls
  prompt_d_random_few_shot_with_dict_8exmpls
  prompt_d_random_few_shot_with_dict_16exmpls
  prompt_d_random_few_shot_with_dict_32exmpls
  prompt_d_balanced_few_shot_with_dict_2exmpls
  prompt_d_balanced_few_shot_with_dict_4exmpls

In [ ]:
# ─────────────────────────────────────────────
# Настройка логирования
# ─────────────────────────────────────────────
class ColaFormatter(logging.Formatter):
    COLORS = {
        logging.DEBUG:    "\033[37m",
        logging.INFO:     "\033[36m",
        logging.WARNING:  "\033[33m",
        logging.ERROR:    "\033[31m",
        logging.CRITICAL: "\033[35m",
    }
    RESET = "\033[0m"

    def format(self, record):
        color = self.COLORS.get(record.levelno, self.RESET)
        record.levelname = f"{color}{record.levelname:<8}{self.RESET}"
        return super().format(record)


def setup_logger(
    name: str,
    log_file: str,
    console_level: int = logging.INFO,
    file_level: int    = logging.DEBUG,
) -> logging.Logger:
    logger = logging.getLogger(name)
    logger.setLevel(logging.DEBUG)

    if logger.handlers:
        logger.handlers.clear()

    file_fmt = logging.Formatter(
        fmt="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S",
    )
    console_fmt = ColaFormatter(
        fmt="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
        datefmt="%H:%M:%S",
    )

    file_path = LOG_DIR / log_file
    fh = logging.FileHandler(file_path, encoding="utf-8", mode="a")
    fh.setLevel(file_level)
    fh.setFormatter(file_fmt)
    logger.addHandler(fh)

    ch = logging.StreamHandler(sys.stdout)
    ch.setLevel(console_level)
    ch.setFormatter(console_fmt)
    logger.addHandler(ch)

    buffer = StringIO()
    bh = logging.StreamHandler(buffer)
    bh.setLevel(logging.DEBUG)
    bh.setFormatter(file_fmt)
    logger.addHandler(bh)

    logger.buffer = buffer
    return logger


logger       = setup_logger("main",  f"run_{RUN_ID}.log",          console_level=logging.INFO)
api_logger   = setup_logger("api",   f"api_{RUN_ID}.log",          console_level=logging.WARNING)
parse_logger = setup_logger("parse", f"parse_errors_{RUN_ID}.log", console_level=logging.WARNING)

logger.info(f"Логирование настроено | RUN_ID={RUN_ID}")
logger.info(f"Логи сохраняются в: {LOG_DIR}")
logger.info(f"Google Drive: {'подключён' if USE_DRIVE else 'не используется'}")


# ─────────────────────────────────────────────
# Утилиты для работы с логами
# ─────────────────────────────────────────────
def show_logs(logger_name: str = "main", tail: int = 50):
    log = logging.getLogger(logger_name)
    if not hasattr(log, "buffer"):
        print("Буфер не найден")
        return
    lines = log.buffer.getvalue().splitlines()
    print(f"\n=== Последние {tail} строк лога [{logger_name}] ===")
    for line in lines[-tail:]:
        print(line)


def show_log_files():
    print(f"\n=== Файлы логов в {LOG_DIR} ===")
    files = sorted(LOG_DIR.glob("*.log"))
    if not files:
        print("  (пусто)")
        return
    for f in files:
        size_kb = f.stat().st_size / 1024
        print(f"  {f.name:<45} {size_kb:>8.1f} KB")


def download_logs():
    try:
        from google.colab import files
        log_files = sorted(LOG_DIR.glob("*.log"))
        if not log_files:
            print("Нет файлов для скачивания")
            return
        for f in log_files:
            print(f"Скачиваем: {f.name}")
            files.download(str(f))
    except ImportError:
        print("Функция доступна только в Google Colab")


def clear_log_buffer(logger_name: str = "main"):
    log = logging.getLogger(logger_name)
    if hasattr(log, "buffer"):
        log.buffer.truncate(0)
        log.buffer.seek(0)
        print(f"Буфер [{logger_name}] очищен")


21:19:36 | INFO     | main | Логирование настроено | RUN_ID=20260605_211934


INFO    :main:Логирование настроено | RUN_ID=20260605_211934


21:19:36 | INFO     | main | Логи сохраняются в: /content/drive/MyDrive/logs


INFO    :main:Логи сохраняются в: /content/drive/MyDrive/logs


21:19:36 | INFO     | main | Google Drive: подключён


INFO    :main:Google Drive: подключён


# АСИНХРОННЫЕ ЗАПРОСЫ

In [ ]:
async def send_one_request(
    client,
    model_name: str,
    messages: list,
    item_id: str = "unknown",
):
    api_logger.debug(
        f"[{item_id}] → Запрос | "
        f"prompt_len={sum(len(m['content']) for m in messages)}"
    )
    start = time.time()
    try:
        response = await client.chat.completions.create(
            model=model_name,
            messages=messages,
            max_tokens=2048,
            temperature=0.0,
            seed=42,
        )
        elapsed           = time.time() - start
        content           = response.choices[0].message.content
        prompt_tokens     = response.usage.prompt_tokens
        completion_tokens = response.usage.completion_tokens
        total_tokens      = prompt_tokens + completion_tokens

        # ПРАВКА 2: логируем токены на каждый объект
        api_logger.debug(
            f"[{item_id}] ← Ответ | "
            f"time={elapsed:.2f}s | "
            f"prompt_tokens={prompt_tokens} | "
            f"completion_tokens={completion_tokens} | "
            f"total_tokens={total_tokens} | "
            f"preview={content[:60].replace(chr(10),' ')!r}"
        )
        # INFO-уровень: краткая запись токенов (видна в консоли)
        api_logger.info(
            f"[{item_id}] tokens: prompt={prompt_tokens} "
            f"completion={completion_tokens} total={total_tokens}"
        )

        return {
            "response":          content,
            "time":              elapsed,
            "prompt_tokens":     prompt_tokens,
            "completion_tokens": completion_tokens,
            "total_tokens":      total_tokens,  # ← добавлено для агрегации
        }

    except RateLimitError as e:
        elapsed = time.time() - start
        api_logger.warning(f"[{item_id}] RateLimitError | time={elapsed:.2f}s | {e}")
        raise

    except APIConnectionError as e:
        elapsed = time.time() - start
        api_logger.error(f"[{item_id}] APIConnectionError | time={elapsed:.2f}s | {e}")
        raise

    except APIStatusError as e:
        elapsed = time.time() - start
        api_logger.error(
            f"[{item_id}] APIStatusError | "
            f"time={elapsed:.2f}s | "
            f"status={e.status_code} | "
            f"body={str(e.body)[:200]}"
        )
        raise

    except Exception as e:
        elapsed = time.time() - start
        api_logger.error(
            f"[{item_id}] UnexpectedError | "
            f"time={elapsed:.2f}s | "
            f"{type(e).__name__}: {e}"
        )
        raise


async def process_with_semaphore(
    client,
    model_name: str,
    messages: list,
    item_id: str = "unknown",
):
    async with semaphore:
        return await send_one_request(client, model_name, messages, item_id=item_id)


def parse_response(raw: str, item_id: str):
    try:
        cleaned = re.sub(r"```(?:json)?|```", "", raw).strip()
        data    = json.loads(cleaned)
        result  = {
            "id":               str(data.get("id", item_id)),
            "has_drug_mention": bool(data.get("has_drug_mention", False)),
        }
        api_logger.debug(
            f"[{item_id}] Парсинг OK | "
            f"has_drug_mention={result['has_drug_mention']}"
        )
        return result

    except json.JSONDecodeError as e:
        parse_logger.error(
            f"[{item_id}] JSONDecodeError: {e} | "
            f"raw={raw[:200].replace(chr(10),' ')!r}"
        )
        return None

    except Exception as e:
        parse_logger.error(
            f"[{item_id}] ParseError: {type(e).__name__}: {e} | "
            f"raw={raw[:200].replace(chr(10),' ')!r}"
        )
        return None


# ОЦЕНКА МЕТРИК

In [ ]:
def evaluate_results(
    results_file: str,
    df_truth: pd.DataFrame,
    label: str,
    truth_label_col: str = "message_label",  # имя колонки с истинной меткой
):
    if not Path(results_file).exists():
        print(f"[{label}] файл {results_file} не найден, пропускаю")
        return None

    with open(results_file) as f:
        preds = json.load(f)

    df_pred = pd.DataFrame(preds)
    if df_pred.empty:
        print(f"[{label}] файл пустой, пропускаю")
        return None

    df_pred = df_pred.drop_duplicates(subset="id", keep="last")
    df_pred["id"] = df_pred["id"].astype(str)

    df_truth_local = df_truth.copy()
    df_truth_local["composite_id"] = (
        df_truth_local["session_id"].astype(str) + "___" +
        df_truth_local.index.astype(str)
    )

    # Диагностика выравнивания
    expected_ids = set(df_truth_local["composite_id"])
    actual_ids   = set(df_pred["id"])
    missing = expected_ids - actual_ids
    extra   = actual_ids   - expected_ids
    if missing or extra:
        print(
            f"[{label}] ВНИМАНИЕ: "
            f"пропущено id из truth: {len(missing)}, "
            f"лишних id в pred: {len(extra)}"
        )
        if extra:
            print(f"  пример лишних id: {list(extra)[:3]}")
        if missing:
            print(f"  пример пропущенных id: {list(missing)[:3]}")

    df_merged = df_truth_local.merge(
        df_pred, left_on="composite_id", right_on="id", how="inner"
    )

    if len(df_merged) == 0:
        print(f"[{label}] нет совпадений по id, пропускаю.")
        return None

    if len(df_merged) != len(df_truth_local):
        print(
            f"[{label}] предупреждение: "
            f"смержилось {len(df_merged)} из {len(df_truth_local)} строк"
        )

    y_true = (df_merged[truth_label_col] == "illegal").astype(int)
    y_pred = df_merged["has_drug_mention"].astype(int)

    # ПРАВКА 1: считаем binary F1 и macro F1
    metrics = {
        "version":    label,
        "n":          len(df_merged),
        "accuracy":   accuracy_score(y_true, y_pred),
        "precision":  precision_score(y_true, y_pred, zero_division=0),
        "recall":     recall_score(y_true, y_pred, zero_division=0),
        "f1_binary":  f1_score(y_true, y_pred, average="binary",  zero_division=0),
        "f1_macro":   f1_score(y_true, y_pred, average="macro",   zero_division=0),
    }

    print(f"\n=== Промпт {label} (n={metrics['n']}) ===")
    print(f"  accuracy   : {metrics['accuracy']:.4f}")
    print(f"  precision  : {metrics['precision']:.4f}")
    print(f"  recall     : {metrics['recall']:.4f}")
    print(f"  f1 (binary): {metrics['f1_binary']:.4f}")
    print(f"  f1 (macro) : {metrics['f1_macro']:.4f}")   # ← ПРАВКА 1

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    print(f"\n  Матрица ошибок [строки=truth (legal, illegal), столбцы=pred]:")
    print(pd.DataFrame(
        cm,
        index=["truth_legal", "truth_illegal"],
        columns=["pred_legal", "pred_illegal"],
    ))

    print(f"\n  classification_report:")
    print(classification_report(
        y_true, y_pred,
        target_names=["legal", "illegal"],
        zero_division=0,
    ))

    return metrics


# ГЛАВНАЯ ФУНКЦИЯ

In [ ]:
async def run_variant(client, variant_name: str, items: list):
    """
    Классифицирует все items одним вариантом промпта.
    ПРАВКА 2: логирует сводную статистику токенов по варианту.
    """
    logger.info(f"{'='*50}")
    logger.info(f"Запускаем вариант: {variant_name}")
    logger.info(f"Записей: {len(items)}")

    builder = PROMPT_VARIANTS[variant_name]

    # Передаём item_id в process_with_semaphore для детального логирования
    tasks = [
        asyncio.create_task(
            process_with_semaphore(
                client,
                MODEL_NAME,
                builder(item),
                item_id=item["id"],   # ← передаём id
            )
        )
        for item in items
    ]

    start       = time.time()
    raw_results = await tqdm.gather(*tasks, desc=variant_name)
    total_time  = time.time() - start

    # Собираем только успешные ответы
    valid = [r for r in raw_results if not isinstance(r, Exception)]

    # ── ПРАВКА 2: сводная статистика токенов по варианту ──────────────
    if valid:
        sum_prompt     = sum(r["prompt_tokens"]     for r in valid)
        sum_completion = sum(r["completion_tokens"] for r in valid)
        sum_total      = sum(r["total_tokens"]      for r in valid)
        avg_prompt     = sum_prompt     / len(valid)
        avg_completion = sum_completion / len(valid)
        avg_total      = sum_total      / len(valid)
        avg_time       = sum(r["time"]  for r in valid) / len(valid)

        token_summary = (
            f"[{variant_name}] СВОДКА ТОКЕНОВ | "
            f"запросов={len(valid)}/{len(items)} | "
            f"prompt: sum={sum_prompt} avg={avg_prompt:.1f} | "
            f"completion: sum={sum_completion} avg={avg_completion:.1f} | "
            f"total: sum={sum_total} avg={avg_total:.1f} | "
            f"avg_time={avg_time:.2f}s | "
            f"total_time={total_time:.2f}s"
        )
        # Пишем в оба логгера: api и main
        api_logger.info(token_summary)
        logger.info(token_summary)

        print(f"\n--- Сводка токенов [{variant_name}] ---")
        print(f"  Успешных запросов : {len(valid)} / {len(items)}")
        print(f"  prompt_tokens     : sum={sum_prompt}  avg={avg_prompt:.1f}")
        print(f"  completion_tokens : sum={sum_completion}  avg={avg_completion:.1f}")
        print(f"  total_tokens      : sum={sum_total}  avg={avg_total:.1f}")
        print(f"  Среднее время/запрос : {avg_time:.2f} сек")
        print(f"  Общее время          : {total_time:.2f} сек")
    else:
        logger.warning(f"[{variant_name}] Нет успешных ответов!")
    # ── конец правки 2 ─────────────────────────────────────────────────

    # Парсим ответы
    parsed = []
    for item, raw in zip(items, raw_results):
        if isinstance(raw, Exception):
            logger.warning(f"  ⚠️  Ошибка запроса для id={item['id']}: {raw}")
            continue
        result = parse_response(raw["response"], item["id"])
        if result:
            parsed.append(result)

    # Сохраняем результаты
    output_file = f"results_{variant_name}.json"
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(parsed, f, ensure_ascii=False, indent=2)
    logger.info(f"Сохранено {len(parsed)} результатов → {output_file}")

    # Формируем stats для возврата
    stats = None
    if valid:
        stats = {
            "time":       {"total": total_time,
                           "avg":   sum(r["time"] for r in valid) / len(valid),
                           "min":   min(r["time"] for r in valid),
                           "max":   max(r["time"] for r in valid)},
            "prompt":     {"sum": sum_prompt,     "avg": avg_prompt,
                           "min": min(r["prompt_tokens"]     for r in valid),
                           "max": max(r["prompt_tokens"]     for r in valid)},
            "completion": {"sum": sum_completion, "avg": avg_completion,
                           "min": min(r["completion_tokens"] for r in valid),
                           "max": max(r["completion_tokens"] for r in valid)},
            "total":      {"sum": sum_total,      "avg": avg_total,
                           "min": min(r["total_tokens"]      for r in valid),
                           "max": max(r["total_tokens"]      for r in valid)},
        }

    return parsed, stats


async def main():
    global semaphore
    semaphore = asyncio.Semaphore(MAX_CONCURRENCY)

    client = AsyncOpenAI(
        api_key=API_KEY,
        base_url=BASE_URL,
    )
    logger.info(f"Модель: {MODEL_NAME}")
    logger.info(f"Записей для классификации (val): {len(input_jsons)}")

    # ── Шаг 1: классификация на валидационной выборке ─────────────────
    logger.info("Шаг 1: классификация всеми вариантами промптов на val.parquet")

    # Накапливаем stats по каждому варианту
    all_stats: dict[str, dict] = {}

    for variant_name in PROMPT_VARIANTS:
        parsed, stats = await run_variant(client, variant_name, input_jsons)
        if stats is not None:
            all_stats[variant_name] = stats

    # ── Шаг 2: метрики на валидационной выборке ───────────────────────
    logger.info("Шаг 2: расчёт метрик на валидационной выборке")
    print(f"\n{'='*60}")
    print("МЕТРИКИ НА ВАЛИДАЦИОННОЙ ВЫБОРКЕ (val.parquet)")
    print(f"{'='*60}")

    all_metrics = []
    for variant_name in PROMPT_VARIANTS:
        m = evaluate_results(
            results_file=f"results_{variant_name}.json",
            df_truth=df_val,
            label=variant_name,
            truth_label_col="message_label",
        )
        if m is not None:
            all_metrics.append(m)

    if all_metrics:
        df_metrics = (
            pd.DataFrame(all_metrics)
            .set_index("version")
            .round(4)
        )
        print("\n=== Сводная таблица ===")
        print(df_metrics)

        metrics_file = f"metrics_val_{RUN_ID}.csv"
        df_metrics.to_csv(metrics_file)
        logger.info(f"Метрики сохранены → {metrics_file}")

        best_variant = df_metrics["f1_macro"].idxmax()
        best_f1      = df_metrics.loc[best_variant, "f1_macro"]
        logger.info(
            f"Лучший вариант по f1_macro: {best_variant} "
            f"(f1_macro={best_f1:.4f})"
        )
        print(
            f"\n🏆 Лучший вариант по f1_macro: "
            f"{best_variant} (f1_macro={best_f1:.4f})"
        )

    # ── Шаг 3: сводный df по времени и токенам ────────────────────────
    if all_stats:
        rows = []
        for variant_name, s in all_stats.items():
            rows.append({
                "variant":            variant_name,
                # время
                "time_total_s":       round(s["time"]["total"], 2),
                "time_avg_s":         round(s["time"]["avg"],   2),
                "time_min_s":         round(s["time"]["min"],   2),
                "time_max_s":         round(s["time"]["max"],   2),
                # prompt токены
                "prompt_sum":         s["prompt"]["sum"],
                "prompt_avg":         round(s["prompt"]["avg"], 1),
                "prompt_min":         s["prompt"]["min"],
                "prompt_max":         s["prompt"]["max"],
                # completion токены
                "completion_sum":     s["completion"]["sum"],
                "completion_avg":     round(s["completion"]["avg"], 1),
                "completion_min":     s["completion"]["min"],
                "completion_max":     s["completion"]["max"],
                # total токены
                "total_tokens_sum":   s["total"]["sum"],
                "total_tokens_avg":   round(s["total"]["avg"], 1),
                "total_tokens_min":   s["total"]["min"],
                "total_tokens_max":   s["total"]["max"],
            })

        df_time = pd.DataFrame(rows).set_index("variant")

        print(f"\n{'='*60}")
        print("СВОДКА ПО ВРЕМЕНИ И ТОКЕНАМ")
        print(f"{'='*60}")

        # Время отдельно для читаемости
        print("\n--- Время выполнения (сек) ---")
        print(df_time[["time_total_s", "time_avg_s", "time_min_s", "time_max_s"]])

        # Токены отдельно
        print("\n--- Токены: prompt ---")
        print(df_time[["prompt_sum", "prompt_avg", "prompt_min", "prompt_max"]])

        print("\n--- Токены: completion ---")
        print(df_time[["completion_sum", "completion_avg", "completion_min", "completion_max"]])

        print("\n--- Токены: total ---")
        print(df_time[["total_tokens_sum", "total_tokens_avg", "total_tokens_min", "total_tokens_max"]])

        # Сохраняем
        time_file = f"time_tokens_{RUN_ID}.csv"
        df_time.to_csv(time_file)
        logger.info(f"Сводка по времени и токенам сохранена → {time_file}")
        print(f"\n💾 Сохранено → {time_file}")

    # ── Финальная сводка ──────────────────────────────────────────────
    logger.info("=" * 60)
    logger.info("ФИНАЛЬНАЯ СВОДКА — см. строки 'ТОКЕНЫ' в api_*.log")
    logger.info("=" * 60)



if __name__ == "__main__":
    nest_asyncio.apply()
    asyncio.run(main())

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
'''
# Последние 30 строк основного лога
show_logs("main", tail=30)

# Все ошибки парсинга
show_logs("parse", tail=100)

# Список файлов логов
show_log_files()

# Скачать логи в браузер
download_logs()
'''

'\n# Последние 30 строк основного лога\nshow_logs("main", tail=30)\n\n# Все ошибки парсинга\nshow_logs("parse", tail=100)\n\n# Список файлов логов\nshow_log_files()\n\n# Скачать логи в браузер\ndownload_logs()\n'